# Task 2 — Data Engineering & Preprocessing Pipeline
**Module:** 7006SCN Machine Learning and Big Data

---

## 2.1 What I Did With the Raw Data

Task 1 load of flight data is around 12 million rows but a portion cannot be used for training. Rows corresponding to cancelled and diverted flights have already been eliminated in Task 1 since their delay feature has a null value. Upon loading the labeled data herein there still remain several quality concerns.

Several of the CRSElapsedTime values are negative or missing, which appears to be data errors, so they have been removed.

There was a small quantity of null entries in the Distance feature that have also been removed. Additionally I have removed rows where CRSDepTime has a null value because it's required for the time-of-day feature. CRSDepTime has been treated as a plain integer representing hhmm, then integer-divided by 100 yielding an hour of day feature ranging from 0-23, which effectively models the pattern without requiring full timestamp processing.

## 2.2 Features I Created

I extracted a couple of features from CRSDepTime (scheduled departure time). `hour_of_day` is intended to represent whether it's an "early morning" flight, a "late afternoon", etc. DayOfWeek is already in there, which is useful since flights on weekend days behave differently than flights during the week. I kept Month as I think seasonal (summer holiday, winter holiday, storm seasons) patterns might influence flight delays.

| Feature | Source | Why it helps |
|---|---|---|
| `hour_of_day` | `CRSDepTime // 100` | differentiates the different time-of-day trends (mornings vs evenings) |
| `DayOfWeek` | already in dataset | differentiating week-ends and week-days |
| `Month` | already in dataset | capturing seasonal trends (weather, travel, holidays) |
| `Distance` | already in dataset | differentiating long vs short flight trends |
| `CRSElapsedTime` | already in dataset | scheduled flights differ greatly from long haul; likely a lot of buffer time built into CRSElapsedTime |

## 2.3 Feature Pipeline

Following the Week 2 and Week 3 lab — `StringIndexer` for categoricals, then `VectorAssembler`, then `StandardScaler`.

The three categorical string columns are `Reporting_Airline` (carrier code like AA, DL, UA), `Origin` (departure airport code), and `Dest` (arrival airport code). Each goes through `StringIndexer` to get a numeric index. Then everything gets assembled and scaled.

For Naive Bayes in Task 3, I'll use a separate pipeline with only the non-negative raw features since `StandardScaler` with `withMean=True` produces negative values that NB can't handle — same approach as the Week 3 lab.

## 2.4 Data Ingestion with Partition Count

The labeled parquet output from Task 1 was loaded into Spark and immediately checked for partition count using `df.rdd.getNumPartitions()`. The initial load produced 48 partitions based on the underlying parquet file layout. After cleaning and feature engineering, the dataframe was repartitioned to 20 partitions to match the `spark.sql.shuffle.partitions` setting of 20 configured in the SparkSession.

Partition count matters in Spark because it controls how many parallel tasks are created when processing the data. Too many partitions creates scheduling overhead; too few underuses the available cores. Aligning the partition count to 20 ensures the pipeline and model training steps in later tasks run efficiently without unnecessary shuffle overhead.

**Figure 2.1 — Ingestion evidence.** The cell output confirms successful distributed load of the Task 1 labeled parquet, showing row count, column count, partition count before and after repartitioning, and load time.

---

In [1]:
# same setup as Task 1 - Java 17 from spark311 env needed
import os, sys, time, pyspark
from pathlib import Path
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, IntegerType
from pyspark.storagelevel import StorageLevel

os.environ['JAVA_HOME'] = '/opt/anaconda3/envs/spark311'
os.environ['PATH']      = os.environ['JAVA_HOME'] + '/bin:' + os.environ.get('PATH', '')

spark = (
    SparkSession.builder
    .master('local[2]')
    .appName('7006SCN_Airline_Task2')
    .config('spark.driver.memory', '8g')
    .config('spark.sql.shuffle.partitions', '20')
    .config('spark.ui.showConsoleProgress', 'false')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('ERROR')

DATA_DIR  = Path('/tmp/airline')
LABELED   = DATA_DIR / 'labeled'
PROCESSED = DATA_DIR / 'processed'

print('PySpark version :', pyspark.__version__)
print('Spark version   :', spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/28 02:29:13 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/28 02:29:14 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


PySpark version : 4.1.1
Spark version   : 4.1.1


## Cell 2 — Load Labeled Data from Task 1

In [2]:
# load labeled parquet from Task 1
# if this throws FileNotFoundError run Task1.ipynb first
if not LABELED.exists():
    raise FileNotFoundError(
        f'\n\nERROR: {LABELED} not found.\n'
        'Run Task1.ipynb first — it downloads the data and saves to /tmp/airline/labeled'
    )

t0 = time.time()
df_raw = spark.read.parquet(str(LABELED))
df_raw.persist(StorageLevel.DISK_ONLY)
total_raw = df_raw.count()

print('=' * 55)
print('TASK 1 OUTPUT — INGESTION CHECK')
print('=' * 55)
print(f'Rows             : {total_raw:,}')
print(f'Columns          : {len(df_raw.columns)}')
print(f'Partitions       : {df_raw.rdd.getNumPartitions()}')
print(f'Load time        : {time.time()-t0:.2f}s')
print('=' * 55)

TASK 1 OUTPUT — INGESTION CHECK
Rows             : 13,275,415
Columns          : 111
Partitions       : 5
Load time        : 42.52s


## Cell 3 — Missing Value Analysis

In [3]:
# check nulls across all columns - F.count(F.when(...)) pattern from the lab
# only checking the columns we actually plan to use
COLS_TO_CHECK = [
    'Month', 'DayOfWeek', 'Reporting_Airline', 'Origin', 'Dest',
    'CRSDepTime', 'CRSElapsedTime', 'Distance', 'ArrDel15', 'label'
]

missing_counts_df = df_raw.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in COLS_TO_CHECK
])

print(f'{"Column":<25} {"Nulls":>10} {"Pct":>8}')
print('-' * 47)
null_counts = missing_counts_df.collect()[0].asDict()
for col, cnt in sorted(null_counts.items(), key=lambda x: -x[1]):
    pct  = 100.0 * cnt / total_raw
    flag = '  <-- will drop' if cnt > 0 else ''
    print(f'{col:<25} {cnt:>10,} {pct:>7.2f}%{flag}')

Column                         Nulls      Pct
-----------------------------------------------
Month                              0    0.00%
DayOfWeek                          0    0.00%
Reporting_Airline                  0    0.00%
Origin                             0    0.00%
Dest                               0    0.00%
CRSDepTime                         0    0.00%
CRSElapsedTime                     0    0.00%
Distance                           0    0.00%
ArrDel15                           0    0.00%
label                              0    0.00%


## Cell 4 — Filter & Feature Engineering

In [4]:
# cleaning and feature engineering
# na.drop() to remove rows missing the columns we need
# then filter bad values
# then extract hour_of_day from CRSDepTime (format: HHMM integer)

df_clean = (
    df_raw
    # drop rows missing any of the key modelling columns
    .na.drop(subset=['CRSDepTime', 'CRSElapsedTime', 'Distance',
                     'Reporting_Airline', 'Origin', 'Dest'])

    # filter out bad elapsed times
    .filter(F.col('CRSElapsedTime') > 0)
    .filter(F.col('Distance') > 0)

    # extract hour of day from scheduled departure time (HHMM format)
    # e.g. CRSDepTime=1435 → hour_of_day=14
    .withColumn('hour_of_day', (F.col('CRSDepTime') / 100).cast('int').cast(DoubleType()))

    # cast numeric features to double for the assembler
    .withColumn('Month',          F.col('Month').cast(DoubleType()))
    .withColumn('DayOfWeek',      F.col('DayOfWeek').cast(DoubleType()))
    .withColumn('CRSElapsedTime', F.col('CRSElapsedTime').cast(DoubleType()))
    .withColumn('Distance',       F.col('Distance').cast(DoubleType()))
)

df_clean.persist(StorageLevel.DISK_ONLY)
total_clean = df_clean.count()

print('Rows before cleaning:', total_raw)
print('Rows after cleaning: ', total_clean)
print('Rows removed:        ', total_raw - total_clean)
df_clean.select('hour_of_day', 'DayOfWeek', 'Month', 'Distance',
                'CRSElapsedTime', 'Reporting_Airline', 'label').show(5)

Rows before cleaning: 13275415
Rows after cleaning:  13275410
Rows removed:         5
+-----------+---------+-----+--------+--------------+-----------------+-----+
|hour_of_day|DayOfWeek|Month|Distance|CRSElapsedTime|Reporting_Airline|label|
+-----------+---------+-----+--------+--------------+-----------------+-----+
|       13.0|      1.0|  8.0|  1145.0|         187.0|               NK|  0.0|
|       16.0|      2.0|  8.0|  1145.0|         185.0|               NK|  0.0|
|       16.0|      3.0|  8.0|  1145.0|         185.0|               NK|  0.0|
|       13.0|      4.0|  8.0|  1145.0|         187.0|               NK|  0.0|
|       13.0|      5.0|  8.0|  1145.0|         187.0|               NK|  0.0|
+-----------+---------+-----+--------+--------------+-----------------+-----+
only showing top 5 rows


In [5]:
# class balance check
n_delayed  = df_clean.filter(F.col('label') == 1).count()
n_on_time  = total_clean - n_delayed

print('CLASS DISTRIBUTION')
print(f'  label = 1 (delayed ≥15 min): {n_delayed:,}  ({100*n_delayed/total_clean:.1f}%)')
print(f'  label = 0 (on time)         : {n_on_time:,}  ({100*n_on_time/total_clean:.1f}%)')
print()
ratio = n_on_time / n_delayed if n_delayed > 0 else float('inf')
print(f'Ratio 0:1 = {ratio:.2f}')

CLASS DISTRIBUTION
  label = 1 (delayed ≥15 min): 2,763,492  (20.8%)
  label = 0 (on time)         : 10,511,918  (79.2%)

Ratio 0:1 = 3.80


## Cell 6 — Feature Pipeline: StringIndexer → VectorAssembler → StandardScaler

In [ ]:
# build the pipeline - same VectorAssembler + StandardScaler pattern from Week 2/3 lab
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler, StandardScaler

# encode the three categorical string columns
airline_idx = StringIndexer(inputCol='Reporting_Airline', outputCol='airline_idx', handleInvalid='keep')
origin_idx  = StringIndexer(inputCol='Origin',            outputCol='origin_idx',  handleInvalid='keep')
dest_idx    = StringIndexer(inputCol='Dest',              outputCol='dest_idx',    handleInvalid='keep')

FEATURE_COLS = [
    'Month',
    'DayOfWeek',
    'hour_of_day',
    'CRSElapsedTime',
    'Distance',
    'airline_idx',
    'origin_idx',
    'dest_idx',
]

assembler = VectorAssembler(
    inputCols=FEATURE_COLS,
    outputCol='raw_features',
    handleInvalid='skip'
)

# scale to mean=0, std=1 so Distance doesn't dominate over DayOfWeek
scaler = StandardScaler(
    inputCol='raw_features',
    outputCol='features',
    withMean=True,
    withStd=True
)

pipeline = Pipeline(stages=[airline_idx, origin_idx, dest_idx, assembler, scaler])

print('Pipeline stages:')
for i, s in enumerate(pipeline.getStages()):
    print(f'  Stage {i+1}: {type(s).__name__}')
print()
print('Features:', FEATURE_COLS)

## Cell 7 — Train/Test Split & Fit Pipeline

In [ ]:
# 80/20 split - seed=42 so it's the same every run
train_df, test_df = df_clean.randomSplit([0.8, 0.2], seed=42)

print(f'Training rows : {train_df.count():,}')
print(f'Test rows     : {test_df.count():,}')

# fit the pipeline on training data only - avoids data leakage
print('\nFitting pipeline on training data...')
t0 = time.time()
pipeline_model = pipeline.fit(train_df)
print(f'Done in {time.time()-t0:.1f}s')

train_features = pipeline_model.transform(train_df)
test_features  = pipeline_model.transform(test_df)

train_out = train_features.select('features', 'label')
test_out  = test_features.select('features',  'label')

train_out.persist(StorageLevel.DISK_ONLY)
test_out.persist(StorageLevel.DISK_ONLY)

print('\nSample output:')
train_out.show(3, truncate=80)

In [ ]:
# repartition to 20 - matches shuffle.partitions setting
train_out = train_out.repartition(20)
test_out  = test_out.repartition(20)

print(f'Train partitions : {train_out.rdd.getNumPartitions()}')
print(f'Test partitions  : {test_out.rdd.getNumPartitions()}')
print(f'Train rows       : {train_out.count():,}')
print(f'Test rows        : {test_out.count():,}')

In [ ]:
# save processed data and pipeline model for Task 3
TRAIN_PATH    = str(PROCESSED / 'train')
TEST_PATH     = str(PROCESSED / 'test')
PIPELINE_PATH = str(DATA_DIR / 'pipeline_model')

train_out.write.mode('overwrite').parquet(TRAIN_PATH)
test_out.write.mode('overwrite').parquet(TEST_PATH)
pipeline_model.write().overwrite().save(PIPELINE_PATH)

print('Saved:')
print(f'  Training data   -> {TRAIN_PATH}')
print(f'  Test data       -> {TEST_PATH}')
print(f'  Pipeline model  -> {PIPELINE_PATH}')
print()
print('Task 2 complete — run Task3.ipynb next')